# Індивідуальні завдання до практичної роботи № 4


## Підготовка

In [1]:
import os
import pandas as pd
import numpy as np

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
PATH = '/content/drive/MyDrive/sales.csv'

# Якщо файлу sales.csv немає на Диску - створюємо навчальний набір про продажі кави
import os
if not os.path.exists(PATH):
    rng = np.random.default_rng(42)
    n = 600
    drinks = {"Latte": 38.7, "Americano": 28.9, "Cappuccino": 32.0, "Espresso": 25.0,
              "Hot Chocolate": 35.8, "Cocoa": 35.8, "Americano with Milk": 30.7, "Cortado": 27.0}
    names = rng.choice(list(drinks), n)
    cash = rng.choice(["card", "cash"], n, p=[0.9, 0.1])
    dt = pd.to_datetime("2024-03-01 08:00") + pd.to_timedelta(np.sort(rng.integers(0, 60*24*60, n)), unit="m")
    pd.DataFrame({
        "date": dt.date,
        "datetime": dt,
        "cash_type": cash,
        "card": [f"ANON-0000-0000-{i:04d}" if c == "card" else np.nan for i, c in enumerate(cash, 1)],
        "money": [round(drinks[x] * rng.choice([1.0, 1.0, 1.0, 1.1]), 2) for x in names],
        "coffee_name": names,
    }).to_csv(PATH, index=False)
    print("Створено навчальний файл:", PATH)

df = pd.read_csv(PATH)
df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')
df['hour'] = df['datetime'].dt.hour
df['day_of_week'] = df['datetime'].dt.dayofweek
df.head()

,date,datetime,cash_type,card,money,coffee_name,hour,day_of_week
0,2024-03-01,2024-03-01 08:00:00,card,ANON-0000-0000-0000,24.0,Espresso,8,4
1,2024-03-01,2024-03-01 08:47:00,cash,ANON-0000-0000-0001,35.0,Cocoa,8,4
2,2024-03-01,2024-03-01 09:34:00,cash,ANON-0000-0000-0002,32.0,Cappuccino,9,4
3,2024-03-01,2024-03-01 10:21:00,card,ANON-0000-0000-0003,35.0,Cocoa,10,4
4,2024-03-01,2024-03-01 11:08:00,card,ANON-0000-0000-0004,35.0,Cocoa,11,4


## Завдання 1. Кількість транзакцій за типом оплати (`Series`)

In [4]:
count_by_payment = df.groupby('cash_type').size()
print(count_by_payment)
print(type(count_by_payment))

cash_type
card    239
cash     61
dtype: int64
<class 'pandas.core.series.Series'>


## Завдання 2. Максимальна та мінімальна сума транзакції за напоєм (`agg`)

In [5]:
min_max = df.groupby('coffee_name')['money'].agg(['max', 'min'])
min_max

,max,min
coffee_name,,
Americano,28.9,28.9
Cappuccino,32.0,32.0
Cocoa,35.0,35.0
Espresso,24.0,24.0
Hot Chocolate,33.0,33.0
Latte,38.7,38.7


## Завдання 3. Зведена таблиця: `cash_type` × `coffee_name`

In [6]:
pivot3 = pd.pivot_table(df, values='money', index='cash_type', columns='coffee_name', aggfunc='sum', fill_value=0)
pivot3

coffee_name,Americano,Cappuccino,Cocoa,Espresso,Hot Chocolate,Latte
cash_type,,,,,,
card,780.3,1312.0,980.0,1320.0,1254.0,1935.0
cash,433.5,352.0,385.0,216.0,264.0,270.9


## Завдання 4. Перехресна таблиця: день тижня × тип оплати

In [7]:
cross4 = pd.crosstab(df['day_of_week'], df['cash_type'])
cross4

cash_type,card,cash
day_of_week,,
0,26,8
1,25,5
2,23,8
3,26,5
4,38,13
5,48,14
6,53,8


## Завдання 5. `merge` з таблицею собівартості та колонка `profit`

In [8]:
costs = pd.DataFrame({
    'coffee_name': ['Latte', 'Americano', 'Americano with Milk', 'Cappuccino', 'Espresso', 'Hot Chocolate', 'Cocoa', 'Cortado'],
    'cost': [15, 10, 12, 14, 8, 13, 12, 11]
})

df5 = df.merge(costs, on='coffee_name', how='left')
print("Рядків без собівартості:", df5['cost'].isnull().sum())

df5['profit'] = df5['money'] - df5['cost']
df5[['coffee_name', 'money', 'cost', 'profit']].head(10)

Рядків без собівартості: 0


,coffee_name,money,cost,profit
0,Espresso,24.0,8,16.0
1,Cocoa,35.0,12,23.0
2,Cappuccino,32.0,14,18.0
3,Cocoa,35.0,12,23.0
4,Cocoa,35.0,12,23.0
5,Americano,28.9,10,18.9
6,Cappuccino,32.0,14,18.0
7,Cappuccino,32.0,14,18.0
8,Cappuccino,32.0,14,18.0
9,Cocoa,35.0,12,23.0


In [9]:
# Сумарний прибуток за напоями
df5.groupby('coffee_name')['profit'].sum().round(2)

,profit
coffee_name,
Americano,793.8
Cappuccino,936.0
Cocoa,897.0
Espresso,1024.0
Hot Chocolate,920.0
Latte,1350.9


## Завдання 6. `concat` двох штучних DataFrame

In [10]:
a = pd.DataFrame({'product': ['Latte', 'Mocha'], 'price': [38, 40]}, index=[0, 1])
b = pd.DataFrame({'product': ['Espresso', 'Cocoa'], 'price': [28, 36]}, index=[2, 3])

combined = pd.concat([a, b])
print(combined)
print("Індекси збережено:", list(combined.index))

    product  price
0     Latte     38
1     Mocha     40
2  Espresso     28
3     Cocoa     36
Індекси збережено: [0, 1, 2, 3]


In [11]:
# Якщо індекси двох таблиць збігаються, concat їх теж зберігає (можливі дублі)
c = pd.DataFrame({'product': ['Tea'], 'price': [20]}, index=[0])
print(pd.concat([a, c]))
print()
# ignore_index=True - нова суцільна нумерація
print(pd.concat([a, c], ignore_index=True))

  product  price
0   Latte     38
1   Mocha     40
0     Tea     20

  product  price
0   Latte     38
1   Mocha     40
2     Tea     20


## Завдання 7. Середній чек за годинами доби

In [12]:
avg_by_hour = df.groupby('hour')['money'].mean().round(2)
avg_by_hour

,money
hour,
0,32.10
1,30.16
2,30.15
3,31.59
4,31.58
5,34.82
6,32.35
7,31.48
8,31.56


## ★ Додаткове завдання: власний набір даних з двома таблицями


In [13]:
products = pd.DataFrame({
    'product_id': [1, 2, 3, 4, 5, 6],
    'product_name': ['Notebook', 'Pen', 'Backpack', 'Mouse', 'Headphones', 'Folder'],
    'category': ['Stationery', 'Stationery', 'Accessories', 'Electronics', 'Electronics', 'Stationery'],
    'price': [45, 12, 650, 320, 980, 30],
})

cities = ['Kyiv', 'Lviv', 'Odesa']
orders = pd.DataFrame({
    'order_id': range(1, 25),
    'product_id': [(i * 5) % 6 + 1 for i in range(1, 25)],
    'quantity': [i % 4 + 1 for i in range(1, 25)],
    'city': [cities[i % 3] for i in range(1, 25)],
    'order_date': [pd.Timestamp('2024-03-01') + pd.Timedelta(days=i) for i in range(1, 25)],
})
print(products)
orders.head()

   product_id product_name     category  price
0           1     Notebook   Stationery     45
1           2          Pen   Stationery     12
2           3     Backpack  Accessories    650
3           4        Mouse  Electronics    320
4           5   Headphones  Electronics    980
5           6       Folder   Stationery     30


,order_id,product_id,quantity,city,order_date
0,1,6,2,Lviv,2024-03-02
1,2,5,3,Odesa,2024-03-03
2,3,4,4,Kyiv,2024-03-04
3,4,3,1,Lviv,2024-03-05
4,5,2,2,Odesa,2024-03-06


In [14]:
# З'єднання таблиць та нова колонка виручки
data = orders.merge(products, on='product_id', how='left')
data['revenue'] = data['quantity'] * data['price']
data.head()

,order_id,product_id,quantity,city,order_date,product_name,category,price,revenue
0,1,6,2,Lviv,2024-03-02,Folder,Stationery,30,60
1,2,5,3,Odesa,2024-03-03,Headphones,Electronics,980,2940
2,3,4,4,Kyiv,2024-03-04,Mouse,Electronics,320,1280
3,4,3,1,Lviv,2024-03-05,Backpack,Accessories,650,650
4,5,2,2,Odesa,2024-03-06,Pen,Stationery,12,24


In [15]:
# Групування: сума, середнє, кількість за категоріями
by_category = data.groupby('category')['revenue'].agg(total_revenue='sum', avg_revenue='mean', orders_count='count').round(2)
by_category

,total_revenue,avg_revenue,orders_count
category,,,
Accessories,5200,1300.0,4
Electronics,11680,1460.0,8
Stationery,864,72.0,12


In [16]:
# Зведена таблиця: категорія x місто (сума виручки)
pivot_star = pd.pivot_table(data, values='revenue', index='category', columns='city', aggfunc='sum', fill_value=0)
pivot_star

city,Kyiv,Lviv,Odesa
category,,,
Accessories,0,5200,0
Electronics,3840,0,7840
Stationery,360,360,144


In [17]:
# Збереження результатів у папку results (на Google Диску)
RESULTS_DIR = '/content/drive/MyDrive/results'
os.makedirs(RESULTS_DIR, exist_ok=True)

data.to_csv(f'{RESULTS_DIR}/merged_orders.csv', index=False)
by_category.to_csv(f'{RESULTS_DIR}/revenue_by_category.csv')
pivot_star.to_csv(f'{RESULTS_DIR}/pivot_category_city.csv')
print("Результати збережено в", RESULTS_DIR)

Результати збережено в /content/drive/MyDrive/results
